# EMBED pretraining CSV study notebook

This notebook is for building a more suitable `embed_pretrain.csv` for SSL pretraining.

Design goals:
- start from the merged EMBED metadata + clinical tables
- avoid label-dependent filtering unless you explicitly want it
- put each optional cleanup/filter in its own cell so you can comment it out
- print both the meaning of the field and the dataset size after every step


## Suggested workflow

1. Run the notebook top-to-bottom once.
2. Inspect the row counts and field distributions after each step.
3. Comment out any filter cell that feels too restrictive for pretraining.
4. Re-run from the `Reset working dataframe` cell downward.
5. Save the final table to `data/embed_pretrain.csv` under the repository root.

A reasonable first pretraining candidate is often:
- keep one row per image
- keep `FinalImageType == 2D`
- optionally standardize `ViewPosition`
- optionally keep only `MLO/CC`
- do **not** require `tissueden` unless you specifically need density labels during pretraining


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
META_CSV = Path("/path/to/Mammo/EMBED/tables/EMBED_OpenData_metadata_reduced.csv")
CLINICAL_CSV = Path("/path/to/Mammo/EMBED/tables/EMBED_OpenData_clinical_reduced.csv")
OUTPUT_CSV = REPO_ROOT / "data" / "embed_pretrain.csv"


In [ ]:
def basic_counts(df: pd.DataFrame):
    image_col = "image_path" if "image_path" in df.columns else "anon_dicom_path"
    stats = {"rows": len(df)}
    if "empi_anon" in df.columns:
        stats["patients"] = df["empi_anon"].nunique()
    if "acc_anon" in df.columns:
        stats["exams"] = df["acc_anon"].nunique()
    if image_col in df.columns:
        stats["unique_images"] = df[image_col].nunique()
    return stats


def print_basic_counts(df: pd.DataFrame, title: str):
    stats = basic_counts(df)
    print(f"=== {title} ===")
    for key, value in stats.items():
        print(f"{key}: {value:,}")
    print()


def print_value_summary(df: pd.DataFrame, column: str, topn: int = 10):
    if column not in df.columns:
        print(f"--- {column} ---")
        print("column not found")
        print()
        return
    print(f"--- {column} ---")
    series = df[column]
    print(f"missing: {int(series.isna().sum()):,}")
    print(series.value_counts(dropna=False).head(topn))
    print()


def preview_rows(df: pd.DataFrame, n: int = 5):
    preferred = [
        "empi_anon",
        "acc_anon",
        "image_path",
        "anon_dicom_path",
        "FinalImageType",
        "ViewPosition",
        "spot_mag",
        "GENDER_DESC",
        "tissueden",
        "SeriesDescription",
    ]
    cols = [col for col in preferred if col in df.columns]
    display(df[cols].head(n))


def show_pretrain_fields(df: pd.DataFrame):
    for column in ["FinalImageType", "GENDER_DESC", "ViewPosition", "tissueden", "spot_mag"]:
        print_value_summary(df, column)


def report_filter(step_name: str, explanation: str, before_df: pd.DataFrame, after_df: pd.DataFrame):
    before = basic_counts(before_df)
    after = basic_counts(after_df)
    print(f"STEP: {step_name}")
    print(explanation)
    dropped = before["rows"] - after["rows"]
    kept_ratio = after["rows"] / max(before["rows"], 1)
    print(f"rows: {before['rows']:,} -> {after['rows']:,} (dropped {dropped:,}, kept {kept_ratio:.2%})")
    if "patients" in before and "patients" in after:
        print(f"patients: {before['patients']:,} -> {after['patients']:,}")
    if "unique_images" in before and "unique_images" in after:
        print(f"unique_images: {before['unique_images']:,} -> {after['unique_images']:,}")
    print()


In [ ]:
df_meta = pd.read_csv(META_CSV, low_memory=False)
print_basic_counts(df_meta, "Image-level metadata table")
preview_rows(df_meta)
show_pretrain_fields(df_meta)


In [ ]:
df_clinical = pd.read_csv(CLINICAL_CSV, low_memory=False)
print_basic_counts(df_clinical, "Exam-level clinical table")
preview_rows(df_clinical)
show_pretrain_fields(df_clinical)


In [ ]:
df_merged = pd.merge(df_meta, df_clinical, on=["empi_anon", "acc_anon"], how="inner")
print_basic_counts(df_merged, "Merged EMBED table before pretraining cleanup")
print("Merged on patient id + accession id. One exam can have multiple findings, so image rows can repeat after the merge.")
print()
preview_rows(df_merged)
show_pretrain_fields(df_merged)


In [ ]:
df_merged = df_merged.copy()
df_merged["image_path"] = (
    df_merged["anon_dicom_path"].astype(str)
    .str.replace("/mnt/NAS2/mammo/anon_dicom/", "", regex=False)
    .str.replace(".dcm", ".png", regex=False)
)
df_merged["study_id"] = df_merged["empi_anon"].astype(str)
df_merged["image_id"] = df_merged["image_path"].astype(str).str.split("/").str[-1]

df_work = df_merged.copy()
print_basic_counts(df_work, "Reset working dataframe from merged raw table")
duplicate_rows = int(df_work.duplicated(subset=["image_path"]).sum())
print(f"duplicate image_path rows: {duplicate_rows:,}")
print("This is usually the first thing to inspect for pretraining, because one image can appear multiple times after the metadata/clinical merge.")


## Optional preprocessing and filter cells

Each cell below is intentionally standalone.

If you do not want a filter, comment out that cell and re-run from the `Reset working dataframe` cell above.


In [ ]:
before_df = df_work.copy()
duplicates_before = int(before_df.duplicated(subset=["image_path"]).sum())

df_work = df_work.drop_duplicates(subset=["image_path"], keep="last").reset_index(drop=True)

report_filter(
    "Drop duplicate rows with the same image_path",
    "Meaning: keep one row per physical image file after the metadata/clinical merge. This is usually recommended for SSL pretraining.",
    before_df,
    df_work,
)
print(f"duplicate image_path rows before: {duplicates_before:,}")
print(f"duplicate image_path rows after: {int(df_work.duplicated(subset=['image_path']).sum()):,}")


In [ ]:
before_df = df_work.copy()

df_work = df_work.copy()
df_work.loc[df_work["ViewPosition"].isna(), "ViewPosition"] = "None"
df_work.loc[df_work["SeriesDescription"].isna(), "SeriesDescription"] = "None"
df_work.loc[df_work["SeriesDescription"].str.contains("XCC", na=False), "ViewPosition"] = "XCC"

images_invalid_view_cc = (df_work["ViewPosition"] == "None") & df_work["SeriesDescription"].str.contains("CC", na=False)
df_work.loc[images_invalid_view_cc, "ViewPosition"] = "CC"

images_invalid_view_mlo = (df_work["ViewPosition"] == "None") & df_work["SeriesDescription"].str.contains("MLO", na=False)
df_work.loc[images_invalid_view_mlo, "ViewPosition"] = "MLO"

report_filter(
    "Infer missing ViewPosition from SeriesDescription",
    "Meaning: recover CC/MLO labels when ViewPosition is missing but the series description still contains the projection name.",
    before_df,
    df_work,
)
print_value_summary(df_work, "ViewPosition", topn=12)


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["FinalImageType"] == "2D"].reset_index(drop=True)

report_filter(
    "Keep only FinalImageType == 2D",
    "Meaning: keep standard 2D mammograms and drop c-view or other derived image types. This is often the cleanest pretraining choice if you want a pure FFDM cohort.",
    before_df,
    df_work,
)
print_value_summary(df_work, "FinalImageType")


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["GENDER_DESC"] == "Female"].reset_index(drop=True)

report_filter(
    "Keep only GENDER_DESC == Female",
    "Meaning: restrict the cohort to female patients only. For mammography pretraining this is often reasonable, but it is not strictly required for SSL itself.",
    before_df,
    df_work,
)
print_value_summary(df_work, "GENDER_DESC")


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["tissueden"].notna()].reset_index(drop=True)

report_filter(
    "Keep only rows with tissueden available",
    "Meaning: require a known breast density label. This is useful for density probe tasks, but it is not required for pure SSL pretraining.",
    before_df,
    df_work,
)
print_value_summary(df_work, "tissueden")


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["tissueden"] < 5].reset_index(drop=True)

report_filter(
    "Keep only tissueden < 5",
    "Meaning: keep standard BI-RADS density values 1..4 and drop any special code outside that range. This is label-driven cleanup, not an SSL requirement.",
    before_df,
    df_work,
)
print_value_summary(df_work, "tissueden")


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["ViewPosition"].isin(["MLO", "CC"])].reset_index(drop=True)

report_filter(
    "Keep only standard screening views: MLO and CC",
    "Meaning: keep the two canonical mammography projections. This usually makes the dataset more homogeneous, but it also removes extra views that might still be useful for SSL.",
    before_df,
    df_work,
)
print_value_summary(df_work, "ViewPosition", topn=12)


In [ ]:
before_df = df_work.copy()
df_work = df_work[df_work["spot_mag"].isna()].reset_index(drop=True)

report_filter(
    "Drop spot or magnification views",
    "Meaning: keep routine mammography images and remove spot/magnification acquisitions. This can make the cohort more consistent, but it is optional for SSL.",
    before_df,
    df_work,
)
print_value_summary(df_work, "spot_mag")


In [ ]:
# before_df = df_work.copy()

# exam_text = (
#     df_work["StudyDescription"].fillna("").astype(str) + " " +
#     df_work["SeriesDescription"].fillna("").astype(str) + " " +
#     df_work["ProtocolName"].fillna("").astype(str) + " " +
#     df_work["desc"].fillna("").astype(str)
# ).str.lower()

# df_work = df_work[exam_text.str.contains("screen", na=False)].reset_index(drop=True)

# report_filter(
#     "Keep only screening exams",
#     "Meaning: keep screening mammograms and drop diagnostic/procedure exams, which may include special compression views or protocol-specific artifacts.",
#     before_df,
#     df_work,
# )

# print_basic_counts(df_work, "After screening-only filter")
# print_value_summary(df_work, "StudyDescription", topn=20)


In [ ]:
print_basic_counts(df_work, "Current candidate pretraining table")
preview_rows(df_work, n=10)
show_pretrain_fields(df_work)

print("If this looks good, run the save cell below. If not, comment out one or more filter cells, reset df_work, and re-run from there.")


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_work.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_work):,} rows to {OUTPUT_CSV.resolve()}")
